# AI Puzzle Solver — RL Training (Colab)

Train the DQN agent on Google Colab with GPU acceleration.

**Steps:**
1. Clone the repo
2. Install dependencies
3. Train (GPU-accelerated)
4. Download trained models

In [ ]:
# 1. Clone your repo (replace with your actual repo URL)
# !git clone https://github.com/YOUR_USER/YOUR_REPO.git
# %cd YOUR_REPO

# OR upload the project files manually and cd into it
# %cd /content/project

In [ ]:
# 2. Install dependencies (no pygame needed for training)
!pip install torch numpy matplotlib -q

In [ ]:
# 3. Check GPU availability
import torch
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

In [ ]:
# 4. Train — Difficulty 1
import os
import sys
import time
import numpy as np

# Add project to path
sys.path.insert(0, '.')

from config import RL_MAX_STEPS, RL_TARGET_UPDATE, DEFAULT_GRID_SIZE
from game.level_generator import LevelGenerator
from solvers.rl_solver import RLSolver

def train_colab(
    episodes=10000,
    grid_size=DEFAULT_GRID_SIZE,
    difficulty=1,
    save_dir='models',
    snapshot_every=500,
    log_every=50,
):
    os.makedirs(save_dir, exist_ok=True)
    snapshot_dir = os.path.join(save_dir, 'snapshots')
    os.makedirs(snapshot_dir, exist_ok=True)

    agent = RLSolver(grid_size=grid_size)
    generator = LevelGenerator()

    model_path = os.path.join(save_dir, f'dqn_grid{grid_size}_d{difficulty}.pt')
    if os.path.exists(model_path):
        agent.load(model_path)
        print(f'Resuming from episode {agent.episodes_done}')

    print(f'Training DQN on {grid_size}x{grid_size}, difficulty {difficulty}')
    print(f'Device: {agent.device}')
    print(f'Episodes: {episodes}')
    print('-' * 60)

    rewards_history = []
    solved_history = []
    loss_history = []
    start_time = time.time()

    for ep in range(episodes):
        level = generator.generate(size=grid_size, difficulty=difficulty)
        state = level.copy()
        episode_reward = 0.0
        losses = []

        for step in range(RL_MAX_STEPS):
            obs = agent._pad_observation(state)
            action = agent.select_action(state, training=True)
            next_state, reward, done = state.step(action)
            next_obs = agent._pad_observation(next_state)

            agent.memory.push(obs, action, reward, next_obs, done)
            episode_reward += reward

            loss = agent.train_step()
            if loss is not None:
                losses.append(loss)

            state = next_state
            if done:
                break

        agent.decay_epsilon()
        rewards_history.append(episode_reward)
        solved_history.append(1 if state.won else 0)
        if losses:
            loss_history.append(np.mean(losses))

        if (ep + 1) % RL_TARGET_UPDATE == 0:
            agent.update_target()

        if (ep + 1) % log_every == 0:
            recent_r = rewards_history[-log_every:]
            recent_s = solved_history[-log_every:]
            elapsed = time.time() - start_time
            print(
                f'Ep {agent.episodes_done:5d} | '
                f'Reward: {np.mean(recent_r):7.1f} | '
                f'Solved: {np.mean(recent_s)*100:5.1f}% | '
                f'Eps: {agent.epsilon:.3f} | '
                f'Time: {elapsed:.0f}s'
            )

        if (ep + 1) % 100 == 0:
            agent.save(model_path)

        if (ep + 1) % snapshot_every == 0:
            snap = os.path.join(snapshot_dir, f'dqn_grid{grid_size}_d{difficulty}_ep{agent.episodes_done}.pt')
            agent.save(snap)

    agent.save(model_path)
    print(f'\nDone. Model saved to {model_path}')
    return rewards_history, solved_history, loss_history

rewards, solved, losses = train_colab(
    episodes=10000,
    grid_size=9,
    difficulty=1,
    snapshot_every=500,
)

In [ ]:
# 5. Plot training curves
import matplotlib.pyplot as plt

fig, (ax1, ax2, ax3) = plt.subplots(1, 3, figsize=(15, 4))

window = 50
smoothed_r = np.convolve(rewards, np.ones(window)/window, mode='valid')
ax1.plot(smoothed_r, color='#ff6432')
ax1.set_title('Episode Reward')
ax1.set_xlabel('Episode')

smoothed_s = np.convolve(solved, np.ones(window)/window, mode='valid') * 100
ax2.plot(smoothed_s, color='#64ff64')
ax2.set_title('Solve Rate (%)')
ax2.set_xlabel('Episode')

if losses:
    ax3.plot(losses, color='#00c8ff', alpha=0.5)
    ax3.set_title('Training Loss')
    ax3.set_xlabel('Episode')

plt.suptitle('DQN Training Progress', fontweight='bold')
plt.tight_layout()
plt.savefig('models/training_curves.png', dpi=150)
plt.show()

In [ ]:
# 6. (Optional) Train on difficulty 3 too
rewards3, solved3, losses3 = train_colab(
    episodes=10000,
    grid_size=9,
    difficulty=3,
    snapshot_every=500,
)

In [ ]:
# 7. List all saved models and snapshots
import glob
print('Models:')
for f in sorted(glob.glob('models/**/*.pt', recursive=True)):
    size_mb = os.path.getsize(f) / 1024 / 1024
    print(f'  {f} ({size_mb:.1f} MB)')

In [ ]:
# 8. Download models to your local machine
# Option A: Download as zip
!zip -r models.zip models/
from google.colab import files
files.download('models.zip')

# Option B: Push to your repo
# !git add models/ && git commit -m 'Add trained RL models' && git push